In [55]:
import numpy as np
import pandas as pd
import pickle
print(np.__version__)
print(pd.__version__)
print(pickle.__file__)

2.3.5
3.0.1
C:\Users\krish\AppData\Local\Programs\Python\Python311\Lib\pickle.py


In [105]:
import numpy as np

np.random.seed(42)

# 100,000 is plenty for testing and will train in under a second
n = 100000 

# 1. Four completely independent features (no multicollinearity)
x1 = np.random.uniform(0, 100, n)
x2 = np.random.uniform(0, 100, n)
x3 = np.random.uniform(0, 100, n)
x4 = np.random.uniform(0, 100, n)

# 2. A microscopic whisper of noise (just to keep the math stable)
noise = np.random.normal(0, 0.01, n)

# 3. Target using clean, obvious weights: [5.0, 3.0, 1.5, 0.8] and a bias of 10
y = (
    5.0 * x1 + 
    3.0 * x2 + 
    1.5 * x3 + 
    0.8 * x4 + 
    10.0 + 
    noise
)

# 4. Stack them up
data = np.column_stack([
    x1,
    x2,
    x3,
    x4,
    y
])

# Split into X and y for your model
X = data[:, :-1]
y = data[:, -1]

In [104]:
class ElasticNet:
    def __init__(self,training_input,real_output,test_percent,learning_rate=0.001):
        # basic info
        self.test_percent=test_percent
        self.train_percent=100-self.test_percent
        self.learning_rate=learning_rate
        self.lamb=0.003
        self.log=100
        self.alpha=0.5
        self.batch_size=4096
        # clean and converting
        self.data_cleaning_conversion(training_input,real_output)
        # spliting the data in train and test
        split = int(self.train_percent*len(self.training_input)/100)
        self.X_train=self.training_input[:split]
        self.Y_train=self.real_output[:split]
        self.X_test=self.training_input[split:]
        self.Y_test=self.real_output[split:]
        # normalization
        self.normalization()
        # shape of our data
        self.X_train_shape=self.X_train.shape
        self.Y_train_shape=self.Y_train.shape
        self.X_test_shape=self.X_test.shape
        self.Y_test_shape=self.Y_test.shape
        # adjusting our data
        self.X_train = np.hstack((self.X_train, np.ones((self.X_train.shape[0],1))))
        self.X_test = np.hstack((self.X_test, np.ones((self.X_test.shape[0],1))))
        # weight model
        weights=np.random.randn(self.X_train_shape[1],1)*0.003
        bias=np.random.rand(1,1)
        self.weights_bias = np.vstack((weights,bias))
        # history
        self.log_history=[]
        
    # methods
    def data_cleaning_conversion(self,training_input,real_output):
        training_input = np.nan_to_num(np.asarray(training_input))
        real_output = np.asarray(real_output).reshape(-1,1)
        self.training_input=training_input
        self.real_output=real_output
        # shuffling the data
        indices = np.random.permutation(len(self.training_input))
        self.training_input = self.training_input[indices]
        self.real_output = self.real_output[indices]

    def normalization(self):
        self.X_mean = np.mean(self.X_train, axis=0)
        self.X_std = np.std(self.X_train, axis=0)
        self.X_std[self.X_std == 0] = 1
        self.X_train = (self.X_train - self.X_mean) / self.X_std
        self.X_test = (self.X_test - self.X_mean) / self.X_std
        
        self.Y_mean = np.mean(self.Y_train)
        self.Y_std = np.std(self.Y_train)
        if self.Y_std == 0:
            self.Y_std = 1
        self.Y_train = (self.Y_train - self.Y_mean) / self.Y_std
        self.Y_test = (self.Y_test - self.Y_mean) / self.Y_std
        
    def prediction(self,X_batch):
        self.Y_pred=X_batch @ self.weights_bias
        
    def objective_function(self,y_batch):
        self.error=self.Y_pred-y_batch
        lasso = np.sum(np.abs(self.weights_bias[:-1]))
        ridge = np.sum(self.weights_bias[:-1]**2)
        self.mean_square_error = np.mean(self.error**2)
        self.total_error = np.mean(self.error**2) + self.lamb*( (self.alpha)*lasso + (1-self.alpha)*ridge)
        
    def parameter_updation_equation(self,X_batch,y_batch):
        y_pred = X_batch @ self.weights_bias
        self.error = y_pred - y_batch
        n = len(X_batch)
        gradient = (2/n) * (X_batch.T @ self.error) 
        w = self.weights_bias[:-1]
        l1_penalty = self.alpha * np.sign(w)
        l2_penalty = 2 * (1 - self.alpha) * w
        gradient[:-1] += self.lamb * (l1_penalty + l2_penalty)
        self.weights_bias -= self.learning_rate * gradient
        
    def batch_generate(self,X,y,batch_size):
        n=X.shape[0]
        for i in range(0,n,batch_size):
            yield X[i:i+batch_size], y[i:i+batch_size]
        

    # user call methods
    """
    implementing mini batch system
    """
    def training(self,epochs=1000):
        best_loss = float("inf")
        patience = 10
        counter = 0
        best_weights=np.zeros_like(self.weights_bias)
        for epoch in range(epochs):
            for X_batch, y_batch in self.batch_generate(self.X_train,self.Y_train,self.batch_size):
                # self.prediction(X_batch)
                # self.objective_function(y_batch)
                self.parameter_updation_equation(X_batch,y_batch)
            val_pred = self.X_test @ self.weights_bias
            val_loss = np.mean((val_pred - self.Y_test)**2)
            if val_loss < best_loss:
                best_loss = val_loss
                best_weights = self.weights_bias.copy()
                counter = 0
            else:
                counter += 1
        
            if counter >= patience:
                self.weights_bias= best_weights
                return epoch
            if epoch > 0 and epoch % self.log == 0:
                self.learning_rate*=0.5
                # self.log_history.append({
                #     "epoch": epoch,
                #     "mse": float(self.mean_square_error),
                #     "total_error":float(self.total_error)
                # })
        self.weights_bias = best_weights
    def evaluate(self):
        y_pred_test = self.X_test @ self.weights_bias
        mse = np.mean((y_pred_test - self.Y_test)**2)
        return mse

    def predict(self,values):
        values = np.array(values)
        if values.ndim == 1:
            values = values.reshape(1,-1)
        values = (values - self.X_mean) / self.X_std
        values = np.hstack((values,np.ones((values.shape[0],1))))
        value=values @ self.weights_bias
        return  value*self.Y_std+self.Y_mean
    def r2_score(self):
        y_pred = self.X_test @ self.weights_bias
        ss_res = np.sum((self.Y_test - y_pred)**2)
        y_mean = np.mean(self.Y_test)
        ss_tot = np.sum((self.Y_test - y_mean)**2)
        if ss_tot == 0:
            return 0.0
        r2 = 1 - (ss_res / ss_tot)
        return r2
    def save_model(self,path="elastic_net_model.pkl"):
        import pickle
        with open(path, "wb") as f:
            self.X_train = None
            self.Y_train = None
            # self.X_test = None
            # self.Y_test = None
            self.log=None
            self.test_percent=None
            self.train_percent=None
            self.X_test_shape=None
            self.Y_test_shape=None
            self.X_train_shape=None
            self.Y_train_shape=None
            self.mean_square_error=None
            self.total_error=None
            self.error=None
            pickle.dump(self, f)
            return f"Saved at {path}"
    @staticmethod
    def load_model(path):
        import pickle
        with open(path, "rb") as f:
            return pickle.load(f)

In [106]:
model = ElasticNet(X,y,test_percent=20)

In [107]:
import time

start = time.perf_counter()

model.training(epochs=250)

end = time.perf_counter()

print("Training time:", end - start, "seconds")

Training time: 0.35042120004072785 seconds


In [108]:
model.evaluate()

np.float64(1.8618120227881114e-05)

In [109]:
model.weights_bias

array([[8.20594194e-01],
       [4.92632015e-01],
       [2.46136690e-01],
       [1.30871151e-01],
       [3.38427681e-04]])

In [110]:
model.save_model()

'Saved at elastic_net_model.pkl'

In [111]:
data[0:5]

array([[ 37.45401188,  58.07790434,  28.2587967 ,  15.70539767,
        426.45297026],
       [ 95.07143064,  52.69716496,  45.86765905,   9.55087659,
        719.89783532],
       [ 73.19939418,  35.10369496,   9.92154979,  13.79391995,
        507.21560348],
       [ 59.86584842,  49.32126579,  44.68370253,  47.34893964,
        562.19609602],
       [ 15.60186404,  36.50966354,  20.30813496,  88.45340225,
        298.77538543]])

In [113]:
print(model.predict(X[0:5]).flatten())
print(y[:5])


[427.09164494 719.35783323 507.65706185 562.14403191 299.68417312]
[426.45297026 719.89783532 507.21560348 562.19609602 298.77538543]


In [114]:
model.r2_score()

np.float64(0.9999812857612039)

## comparison with scikit-learn ElasticNet

In [122]:
import numpy as np

np.random.seed(42)

# 100,000 is plenty for testing and will train in under a second
n = 100000 

# 1. Four completely independent features (no multicollinearity)
x1 = np.random.uniform(0, 100, n)
x2 = np.random.uniform(0, 100, n)
x3 = np.random.uniform(0, 100, n)
x4 = np.random.uniform(0, 100, n)

# 2. A microscopic whisper of noise (just to keep the math stable)
noise = np.random.normal(0, 0.01, n)

# 3. Target using clean, obvious weights: [5.0, 3.0, 1.5, 0.8] and a bias of 10
y = (
    5.0 * x1 + 
    3.0 * x2 + 
    1.5 * x3 + 
    0.8 * x4 + 
    10.0 + 
    noise
)

# 4. Stack them up
data = np.column_stack([
    x1,
    x2,
    x3,
    x4,
    y
])

# Split into X and y for your model
X = data[:, :-1]
y = data[:, -1]

In [124]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# same random seed for reproducibility
np.random.seed(42)

# split dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

# scale features (same as your normalization)
X_scaler = StandardScaler()
X_train_scaled = X_scaler.fit_transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

# scale target like your implementation
y_mean = np.mean(y_train)
y_std = np.std(y_train)
y_train_scaled = (y_train - y_mean) / y_std
y_test_scaled = (y_test - y_mean) / y_std

# elastic net model
model = ElasticNet(
    alpha=0.003,      # regularization strength
    l1_ratio=0.5,     # same as your alpha
    max_iter=1000,
    tol=1e-4,
    fit_intercept=True
)

print("Training Scikit-Learn Model...")
model.fit(X_train_scaled, y_train_scaled)

# prediction
y_pred_scaled = model.predict(X_test_scaled)

# convert back to original scale
y_pred = y_pred_scaled * y_std + y_mean

# evaluation
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- SCIKIT-LEARN RESULTS ---")
print(f"MSE: {mse:.6f}")
print(f"R2 Score: {r2:.6f}")

print("\nFirst 5 Predictions:", np.round(y_pred[:5], 2))
print("First 5 Actuals:    ", np.round(y_test[:5], 2))

Training Scikit-Learn Model...

--- SCIKIT-LEARN RESULTS ---
MSE: 0.574610
R2 Score: 0.999981

First 5 Predictions: [409.24 183.6  744.06 662.95 263.33]
First 5 Actuals:     [408.85 182.27 745.06 663.49 262.21]
